In [7]:
%%writefile streamlit_app.py
import streamlit as st
import tensorflow as tf
import numpy as np
from PIL import Image
import os

# Page Configuration

st.set_page_config(
    page_title="Brain Tumor MRI Image Classification",
    layout="centered"
)

st.title(" Brain Tumor MRI Image Classification")
st.write("Upload an MRI image to predict the tumor type")

# Load Model (Transfer Learning - Best Model)

MODEL_PATH = "transfer_model_best.h5"

@st.cache_resource
def load_model():
    if not os.path.exists(MODEL_PATH):
        st.error(f" Model file not found: {MODEL_PATH}")
        st.stop()
    return tf.keras.models.load_model(MODEL_PATH)

model = load_model()


# Class Names 

class_names = ['glioma', 'meningioma', 'no_tumor', 'pituitary']

# Image Preprocessing

def preprocess_image(image):
    image = image.resize((224, 224))
    image = np.array(image) / 255.0
    image = np.expand_dims(image, axis=0)
    return image


# File Uploader

uploaded_file = st.file_uploader(
    "Choose a Brain MRI Image",
    type=["jpg", "jpeg", "png"]
)

if uploaded_file is not None:
    image = Image.open(uploaded_file).convert("RGB")

    st.image(
        image,
        caption="Uploaded MRI Image",
        use_column_width=True
    )

    # Preprocess
    processed_image = preprocess_image(image)

    # Predict
    predictions = model.predict(processed_image)
    predicted_index = np.argmax(predictions)
    predicted_class = class_names[predicted_index]
    confidence = predictions[0][predicted_index] * 100

    # Results
   
    st.markdown("###  Prediction Result")
    st.success(f"**Tumor Type:** {predicted_class}")
    st.info(f"**Confidence:** {confidence:.2f}%")

    st.markdown("###  Class Probabilities")
    for i, class_name in enumerate(class_names):
        st.write(f"{class_name}: {predictions[0][i]*100:.2f}%")





Overwriting streamlit_app.py
